In [16]:
import geopandas as gpd
import requests
import pandas as pd
import time
from tqdm import tqdm
import json



In [ ]:
ny_state_tracts = gpd.read_file(
    "https://www2.census.gov/geo/tiger/TIGER2020/TRACT/tl_2020_36_tract.zip"
)

# Keep only NYC counties
nyc_counties = ['005', '047', '061', '081', '085']  # Bronx, Brooklyn, Manhattan, Queens, Staten Island
nyc_tracts = ny_state_tracts[ny_state_tracts['COUNTYFP'].isin(nyc_counties)].copy()
nyc_tracts = nyc_tracts.to_crs(epsg=4326)  # Required by Ohsome API




,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,36,047,000700,36047000700,7,Census Tract 7,G5020,S,176774,0,+40.6923505,-073.9973434,"POLYGON ((-74.00154 40.69279, -74.00132 40.693..."
1,36,047,000900,36047000900,9,Census Tract 9,G5020,S,163469,0,+40.6917206,-073.9916018,"POLYGON ((-73.99405 40.6909, -73.99374 40.6915..."
2,36,047,001100,36047001100,11,Census Tract 11,G5020,S,168507,0,+40.6932903,-073.9877087,"POLYGON ((-73.99073 40.69305, -73.99045 40.693..."
3,36,047,001300,36047001300,13,Census Tract 13,G5020,S,293167,0,+40.6976150,-073.9883586,"POLYGON ((-73.99141 40.69863, -73.99131 40.699..."
4,36,047,002000,36047002000,20,Census Tract 20,G5020,S,154138,0,+40.6480407,-074.0159276,"POLYGON ((-74.01867 40.64741, -74.01809 40.647..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5406,36,081,027900,36081027900,279,Census Tract 279,G5020,S,177155,0,+40.7516729,-073.8802222,"POLYGON ((-73.88189 40.75208, -73.88096 40.752..."
5407,36,081,028000,36081028000,280,Census Tract 280,G5020,S,165398,0,+40.6883158,-073.7797191,"POLYGON ((-73.78243 40.69034, -73.7816 40.6906..."
5408,36,081,028100,36081028100,281,Census Tract 281,G5020,S,174441,0,+40.7522203,-073.8825920,"POLYGON ((-73.88431 40.75567, -73.88351 40.755..."
5409,36,081,028200,36081028200,282,Census Tract 282,G5020,S,182472,0,+40.6849947,-073.7738873,"POLYGON ((-73.77709 40.6882, -73.77662 40.6883..."


In [32]:
# List of tract GEOIDs you're interested in
target_tracts=nyc_tracts[nyc_tracts['GEOID'].str.startswith('360610099')].copy()
target_tracts
# Filter tracts to just those
#my_tracts = nyc_tracts[nyc_tracts['GEOID'].isin(target_tracts)].copy()
#my_tracts

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
5046,36,061,009903,36061009903,99.03,Census Tract 99.03,G5020,S,420391,437607,+40.7577195,-074.0024316,"POLYGON ((-74.01403 40.75755, -74.01295 40.759..."
5062,36,061,009901,36061009901,99.01,Census Tract 99.01,G5020,S,391367,576525,+40.7489402,-074.0092348,"POLYGON ((-74.01661 40.74696, -74.01601 40.749..."
5064,36,061,009902,36061009902,99.02,Census Tract 99.02,G5020,S,187462,201043,+40.7525256,-074.0066652,"POLYGON ((-74.01425 40.75666, -74.01378 40.756..."


In [25]:
def query_ohsome_grouped(geojson_poly, amenities=["cafe"], start_year=2009, end_year=2020):
    url = "https://api.ohsome.org/v1/elements/count/groupBy/tag"

    # Build OSM filter
    filter_string = "(" + " or ".join([f"amenity={a}" for a in amenities]) + ") and type:node"

    # Convert polygon to JSON string for form upload
    geojson_str = json.dumps({
        "type": "FeatureCollection",
        "features": [{
            "type": "Feature",
            "geometry": geojson_poly,
            "properties": {}
        }]
    })

    # Payload for form-encoded POST
    payload = {
        "bpolys": geojson_str,
        "filter": filter_string,
        "time": f"{start_year}-01-01/{end_year}-12-31/P1Y",
        "format": "json",
        "groupByKey": "tags"
    }

    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    response = requests.post(url, data=payload, headers=headers)

    if response.status_code == 200:
        return response.json()["groupByResult"]
    else:
        print(f"Error: {response.status_code}, {response.text}")
        return []


In [29]:
target_tracts = nyc_tracts[nyc_tracts['GEOID'] == "36061000100"]


In [30]:
# STEP 4: Loop over tracts and collect data
results = []

for idx, row in tqdm(target_tracts.iterrows(), total=len(target_tracts)):
    tract_id = row['GEOID']
    geom = row['geometry'].__geo_interface__
    geojson_poly = {
        "type": "Polygon" if geom['type'] == 'Polygon' else 'MultiPolygon',
        "coordinates": geom['coordinates']
    }

    try:
        grouped_data = query_ohsome_grouped(
            geojson_poly,
            amenities=["cafe", "restaurant", "gym"]
        )

        for group in grouped_data:
            amenity = group['groupByObject']
            if amenity == "remainder":  # 🚫 skip unrelated POIs
                continue
            for entry in group['result']:
                results.append({
                    "tract": tract_id,
                    "amenity": amenity,
                    "timestamp": entry["timestamp"],
                    "count": entry["value"]
                })

    except Exception as e:
        print(f"Error on tract {tract_id}: {e}")

    time.sleep(1)    


100%|██████████| 1/1 [00:04<00:00,  4.96s/it]


In [31]:
df = pd.DataFrame(results)
df

""
